In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# -*- coding: utf-8 -*-
"""
===============================================================================
ECoG视觉解码 · 2026终极方案：ECoG-CLIP跨模态语义对齐
===============================================================================
- 预训练目标：将ECoG信号映射到CLIP文本语义空间（而非信号重构）
- 教师模型：OpenAI CLIP ViT-B/32文本编码器（固定）
- 预训练数据：所有被试ECoG信号 + 对应类别标签的CLIP文本嵌入
- 下游微调：740样本（从所有被试中随机抽取），仅需训练线性分类头
- 文献依据：Communications Biology 2026 - 脑对齐语义向量[citation:1]
===============================================================================
"""

# ---------- 安装CLIP库 ----------
import subprocess
import sys
import importlib

def install_if_missing(package, import_name=None):
    if import_name is None:
        import_name = package
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

install_if_missing('git+https://github.com/openai/CLIP.git', 'clip')

# ---------- 导入核心库 ----------
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
import matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import random
import os
import glob
from tqdm import tqdm
import clip

# ---------- 设置随机种子 ----------
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# ---------- 1. 路径配置 ----------
BASE_DIR = '/kaggle/input/datasets/cieled/visual-2-ecog/data'
print("数据根目录:", BASE_DIR)

# ---------- 2. 加载所有被试数据（统一到72通道×450点）----------
def load_subject_data(subj_id, target_channels=72, target_time=450):
    mat_path = f'{BASE_DIR}/IFP/{subj_id}/data_{subj_id}_exp1to2_t100to800_sa1.mat'
    if not os.path.exists(mat_path):
        return None, None
    mat = loadmat(mat_path)
    p_all = mat['p_all']
    d_all = mat['d_all'][0]
    n_trials = p_all.shape[0]
    n_channels_orig = len(d_all)
    n_timepoints_orig = d_all[0].shape[1]
    use_channels = min(n_channels_orig, target_channels)
    data = np.zeros((n_trials, target_channels, target_time))
    for ch in range(use_channels):
        signal = d_all[ch]
        if n_timepoints_orig >= target_time:
            data[:, ch, :] = signal[:, :target_time]
        else:
            data[:, ch, :n_timepoints_orig] = signal
    labels = p_all[:, 4]
    return data, labels

# 扫描所有被试
subj_dirs = glob.glob(f'{BASE_DIR}/IFP/subj*')
subj_ids = [os.path.basename(d) for d in subj_dirs]
print(f"发现被试: {subj_ids}")

pretrain_data_list = []
pretrain_label_list = []

for subj in subj_ids:
    data, labels = load_subject_data(subj)
    if data is None:
        continue
    pretrain_data_list.append(data)
    pretrain_label_list.append(labels)

# 合并所有被试数据用于预训练
pretrain_data = np.concatenate(pretrain_data_list, axis=0)
pretrain_labels = np.concatenate(pretrain_label_list, axis=0)
print(f"✅ 预训练总样本数: {pretrain_data.shape[0]} (通道{pretrain_data.shape[1]}×时间{pretrain_data.shape[2]})")

# ---------- 3. 数据标准化 ----------
def standardize_data(data):
    for trial in range(data.shape[0]):
        for ch in range(data.shape[1]):
            signal = data[trial, ch, :]
            mean, std = np.mean(signal), np.std(signal)
            if std > 0:
                data[trial, ch, :] = (signal - mean) / std
    return data

pretrain_data = standardize_data(pretrain_data)

# ---------- 4. 构建小样本微调集（740样本）从所有被试中随机抽取----------
# 标签映射：原始标签1,2,3,4,7 -> 0,1,2,3,4
mapping = {1:0, 2:1, 3:2, 4:3, 7:4}
X_all = torch.FloatTensor(pretrain_data)
y_all = torch.LongTensor([mapping[l] for l in pretrain_labels])

n_finetune = 740
n_val = 125
n_test = len(X_all) - n_finetune - n_val

# 随机打乱索引
indices = np.random.permutation(len(X_all))
finetune_indices = indices[:n_finetune]
val_indices = indices[n_finetune:n_finetune+n_val]
test_indices = indices[n_finetune+n_val:]

finetune_X = X_all[finetune_indices]
finetune_y = y_all[finetune_indices]
val_X = X_all[val_indices]
val_y = y_all[val_indices]
test_X = X_all[test_indices]
test_y = y_all[test_indices]

print(f"✅ 微调集: {finetune_X.shape[0]} 样本")
print(f"✅ 验证集: {val_X.shape[0]} 样本")
print(f"✅ 测试集: {test_X.shape[0]} 样本")

# ---------- 5. 加载CLIP模型 ----------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🔧 使用设备: {device}")

print("📦 加载CLIP模型...")
clip_model, _ = clip.load("ViT-B/32", device=device, jit=False)
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad = False

# ---------- 6. 预先生成所有类别的CLIP文本嵌入 ----------
class_names = ['1', '2', '3', '4', '7']
text_descriptions = [f"an image of class {c}" for c in class_names]  # CLIP提示模板
text_tokens = clip.tokenize(text_descriptions).to(device)

with torch.no_grad():
    text_features = clip_model.encode_text(text_tokens)  # [5, 512]
    text_features = F.normalize(text_features, dim=-1)

print(f"✅ CLIP文本特征已生成，形状: {text_features.shape}")

# ---------- 7. 预训练数据集：ECoG + 对应文本特征 ----------
class ECogCLIPDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.FloatTensor(data)
        # 将原始标签映射到0-4
        self.labels = torch.LongTensor([mapping[l] for l in labels])
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

pretrain_dataset = ECogCLIPDataset(pretrain_data, pretrain_labels)
pretrain_loader = DataLoader(pretrain_dataset, batch_size=64, shuffle=True, drop_last=True)

# ---------- 8. ECoG编码器（适配CLIP特征维度512）----------
class ECoGEncoderCLIP(nn.Module):
    """将ECoG信号编码到CLIP共享语义空间（512维）"""
    def __init__(self, input_channels=72, time_points=450, output_dim=512):
        super().__init__()
        # 空间卷积
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool1 = nn.MaxPool1d(2)  # 450 -> 225
        
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.pool2 = nn.MaxPool1d(2)  # 225 -> 112
        
        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(256)
        self.pool3 = nn.MaxPool1d(2)  # 112 -> 56
        
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, output_dim)
        )
        
    def forward(self, x):
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool3(torch.relu(self.bn3(self.conv3(x))))
        x = self.gap(x).squeeze(-1)
        x = self.fc(x)
        return F.normalize(x, dim=-1)  # L2归一化，与CLIP特征对齐

# ---------- 9. 对比损失（ECoG特征 ↔ CLIP文本特征）----------
def clip_contrastive_loss(ecog_features, labels, text_features):
    """
    ecog_features: [B, 512] 已归一化
    labels: [B] 类别索引 (0-4)
    text_features: [5, 512] 已归一化
    """
    # 计算相似度矩阵 [B, 5]
    logits = ecog_features @ text_features.T / 0.07  # 温度系数
    
    # 每个样本的正样本是对应类别的文本特征
    loss = F.cross_entropy(logits, labels)
    
    # 可选：计算准确率作为监控
    with torch.no_grad():
        pred = logits.argmax(dim=-1)
        acc = (pred == labels).float().mean()
    
    return loss, acc

# ---------- 10. 初始化编码器 ----------
encoder = ECoGEncoderCLIP().to(device)
optimizer = optim.AdamW(encoder.parameters(), lr=1e-3, weight_decay=1e-4)

# ========== 11. ECoG-CLIP跨模态对齐预训练 ==========
ssl_epochs = 30
ssl_losses = []
ssl_accs = []

print("\n🚀 启动ECoG-CLIP跨模态对齐预训练...")
for epoch in range(ssl_epochs):
    encoder.train()
    total_loss = 0
    total_acc = 0
    
    pbar = tqdm(pretrain_loader, desc=f'CLIP Epoch {epoch+1}/{ssl_epochs}', leave=False)
    for batch_x, batch_y in pbar:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # 前向：ECoG -> 特征
        ecog_features = encoder(batch_x)
        
        # 对比损失：ECoG特征 vs 固定文本特征
        loss, acc = clip_contrastive_loss(ecog_features, batch_y, text_features)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        total_acc += acc.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{acc.item():.2f}'})
    
    avg_loss = total_loss / len(pretrain_loader)
    avg_acc = total_acc / len(pretrain_loader)
    ssl_losses.append(avg_loss)
    ssl_accs.append(avg_acc)
    print(f'Epoch {epoch+1:2d}: 平均损失 {avg_loss:.4f}, 对齐准确率 {avg_acc:.2f}')

# 绘制预训练曲线
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(ssl_losses)
plt.title('CLIP Contrastive Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(ssl_accs)
plt.title('ECoG→Text Alignment Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

# 保存预训练编码器
torch.save(encoder.state_dict(), 'ecog_clip_encoder.pth')
print("\n✅ ECoG-CLIP编码器已保存")

# ========== 12. 下游分类微调（740样本，包含所有被试）==========
class CLIPClassifier(nn.Module):
    def __init__(self, encoder, num_classes=5):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(512, num_classes)  # 直接在CLIP特征空间分类
        
    def forward(self, x):
        features = self.encoder(x)  # [B, 512]
        return self.classifier(features)

# 加载预训练编码器
finetune_encoder = ECoGEncoderCLIP().to(device)
finetune_encoder.load_state_dict(torch.load('ecog_clip_encoder.pth', map_location=device))
model = CLIPClassifier(finetune_encoder).to(device)

# 类别权重（可选，用于处理不平衡）
class_counts = np.bincount(finetune_y.numpy())
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)
criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(class_weights).to(device))

# ---------- 两阶段微调 ----------
# 第一阶段：冻结编码器，训练分类头
print("\n🎯 第一阶段：训练分类头（冻结编码器）")
for param in model.encoder.parameters():
    param.requires_grad = False

optimizer_head = optim.AdamW(model.classifier.parameters(), lr=1e-3)
finetune_loader = DataLoader(TensorDataset(finetune_X, finetune_y), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(val_X, val_y), batch_size=64, shuffle=False)

stage1_epochs = 10
best_val_acc = 0

for epoch in range(stage1_epochs):
    model.train()
    correct, total = 0, 0
    for batch_x, batch_y in finetune_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        optimizer_head.zero_grad()
        loss.backward()
        optimizer_head.step()
        _, pred = torch.max(outputs, 1)
        correct += (pred == batch_y).sum().item()
        total += batch_y.size(0)
    train_acc = 100 * correct / total
    
    # 验证
    model.eval()
    correct_val, total_val = 0, 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            _, pred = torch.max(outputs, 1)
            correct_val += (pred == batch_y).sum().item()
            total_val += batch_y.size(0)
    val_acc = 100 * correct_val / total_val
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_clip_classifier.pth')
    
    print(f"Stage1 Epoch {epoch+1:2d}: Train Acc {train_acc:.2f}%, Val Acc {val_acc:.2f}%")

# 第二阶段：全参数微调（低学习率）
print("\n🎯 第二阶段：全参数微调")
for param in model.encoder.parameters():
    param.requires_grad = True

optimizer_full = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer_full, mode='min', patience=3, factor=0.5)

stage2_epochs = 20

for epoch in range(stage2_epochs):
    model.train()
    correct, total = 0, 0
    for batch_x, batch_y in finetune_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        optimizer_full.zero_grad()
        loss.backward()
        optimizer_full.step()
        _, pred = torch.max(outputs, 1)
        correct += (pred == batch_y).sum().item()
        total += batch_y.size(0)
    train_acc = 100 * correct / total
    
    model.eval()
    correct_val, total_val = 0, 0
    val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            _, pred = torch.max(outputs, 1)
            correct_val += (pred == batch_y).sum().item()
            total_val += batch_y.size(0)
    val_acc = 100 * correct_val / total_val
    avg_val_loss = val_loss / len(val_loader)
    
    scheduler.step(avg_val_loss)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_clip_classifier.pth')
    
    print(f"Stage2 Epoch {epoch+1:2d}: Train Acc {train_acc:.2f}%, Val Acc {val_acc:.2f}%")

print(f"\n🏆 最佳验证准确率: {best_val_acc:.2f}%")

# ========== 13. 测试集评估 ==========
model.load_state_dict(torch.load('best_clip_classifier.pth', map_location=device))
model.eval()

test_loader = DataLoader(TensorDataset(test_X, test_y), batch_size=64, shuffle=False)
all_preds, all_true = [], []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x)
        _, pred = torch.max(outputs, 1)
        all_preds.extend(pred.cpu().numpy())
        all_true.extend(batch_y.cpu().numpy())

# 还原原始标签
reverse_mapping = {0:1, 1:2, 2:3, 3:4, 4:7}
orig_true = [reverse_mapping[l] for l in all_true]
orig_pred = [reverse_mapping[l] for l in all_preds]

# 混淆矩阵
cm = confusion_matrix(orig_true, orig_pred, labels=[1,2,3,4,7])
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['1','2','3','4','7'],
            yticklabels=['1','2','3','4','7'])
plt.title('Confusion Matrix (ECoG-CLIP)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

print("\n===== 测试集分类报告 =====")
print(classification_report(orig_true, orig_pred, target_names=['1','2','3','4','7']))
test_acc = np.mean(np.array(orig_true) == np.array(orig_pred))
print(f"✅ ECoG-CLIP最终测试集准确率: {test_acc*100:.2f}%")

torch.save(model.state_dict(), 'ecog_clip_final.pth')
print("\n✅ 最终模型已保存为 ecog_clip_final.pth")

In [ ]:
# -*- coding: utf-8 -*-
"""
===============================================================================
ECoG视觉解码 · 2026终极方案：ECoG-CLIP跨模态语义对齐
===============================================================================
- 预训练目标：将ECoG信号映射到CLIP文本语义空间（而非信号重构）
- 教师模型：OpenAI CLIP ViT-B/32文本编码器（固定）
- 预训练数据：所有被试ECoG信号 + 对应类别标签的CLIP文本嵌入
- 下游微调：740样本（从所有被试中随机抽取），仅需训练线性分类头
- 文献依据：Communications Biology 2026 - 脑对齐语义向量[citation:1]
===============================================================================
"""

# ---------- 安装CLIP库 ----------
import subprocess
import sys
import importlib

def install_if_missing(package, import_name=None):
    if import_name is None:
        import_name = package
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

install_if_missing('git+https://github.com/openai/CLIP.git', 'clip')

# ---------- 导入核心库 ----------
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
import matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import random
import os
import glob
from tqdm import tqdm
import clip

# ---------- 设置随机种子 ----------
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# ---------- 1. 路径配置 ----------
BASE_DIR = '/kaggle/input/datasets/cieled/visual-2-ecog/data'
print("数据根目录:", BASE_DIR)

# ---------- 2. 加载所有被试数据（统一到72通道×450点）----------
def load_subject_data(subj_id, target_channels=72, target_time=450):
    """
    加载单个被试的数据，自动匹配正确的文件名（处理 exp1to1, exp1to2, exp1to3 等变体）
    """
    subj_path = f'{BASE_DIR}/IFP/{subj_id}'
    # 使用通配符匹配数据文件，模式：data_{subj_id}_exp*_t100to800_sa1.mat
    pattern = f'{subj_path}/data_{subj_id}_exp*_t100to800_sa1.mat'
    mat_files = glob.glob(pattern)
    if not mat_files:
        print(f"警告: 未找到 {subj_id} 的数据文件，模式: {pattern}")
        return None, None
    mat_path = mat_files[0]  # 取第一个匹配的文件（通常只有一个）
    print(f"加载 {subj_id} 的数据文件: {os.path.basename(mat_path)}")
    
    mat = loadmat(mat_path)
    p_all = mat['p_all']
    d_all = mat['d_all'][0]
    n_trials = p_all.shape[0]
    n_channels_orig = len(d_all)
    n_timepoints_orig = d_all[0].shape[1]
    use_channels = min(n_channels_orig, target_channels)
    data = np.zeros((n_trials, target_channels, target_time))
    for ch in range(use_channels):
        signal = d_all[ch]
        if n_timepoints_orig >= target_time:
            data[:, ch, :] = signal[:, :target_time]
        else:
            data[:, ch, :n_timepoints_orig] = signal
    labels = p_all[:, 4]
    return data, labels

# 扫描所有被试
subj_dirs = glob.glob(f'{BASE_DIR}/IFP/subj*')
subj_ids = [os.path.basename(d) for d in subj_dirs]
print(f"发现被试: {subj_ids}")

pretrain_data_list = []
pretrain_label_list = []

for subj in subj_ids:
    data, labels = load_subject_data(subj)
    if data is None:
        continue
    pretrain_data_list.append(data)
    pretrain_label_list.append(labels)

# 合并所有被试数据用于预训练
pretrain_data = np.concatenate(pretrain_data_list, axis=0)
pretrain_labels = np.concatenate(pretrain_label_list, axis=0)
print(f"✅ 预训练总样本数: {pretrain_data.shape[0]} (通道{pretrain_data.shape[1]}×时间{pretrain_data.shape[2]})")

# ---------- 3. 数据标准化 ----------
def standardize_data(data):
    for trial in range(data.shape[0]):
        for ch in range(data.shape[1]):
            signal = data[trial, ch, :]
            mean, std = np.mean(signal), np.std(signal)
            if std > 0:
                data[trial, ch, :] = (signal - mean) / std
    return data

pretrain_data = standardize_data(pretrain_data)

# ---------- 4. 构建小样本微调集（740样本）从所有被试中随机抽取----------
# 标签映射：原始标签1,2,3,4,7 -> 0,1,2,3,4
mapping = {1:0, 2:1, 3:2, 4:3, 7:4}
X_all = torch.FloatTensor(pretrain_data)
y_all = torch.LongTensor([mapping[l] for l in pretrain_labels])

n_finetune = 740
n_val = 125
n_test = len(X_all) - n_finetune - n_val

# 随机打乱索引
indices = np.random.permutation(len(X_all))
finetune_indices = indices[:n_finetune]
val_indices = indices[n_finetune:n_finetune+n_val]
test_indices = indices[n_finetune+n_val:]

finetune_X = X_all[finetune_indices]
finetune_y = y_all[finetune_indices]
val_X = X_all[val_indices]
val_y = y_all[val_indices]
test_X = X_all[test_indices]
test_y = y_all[test_indices]

print(f"✅ 微调集: {finetune_X.shape[0]} 样本")
print(f"✅ 验证集: {val_X.shape[0]} 样本")
print(f"✅ 测试集: {test_X.shape[0]} 样本")

# ---------- 5. 加载CLIP模型 ----------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🔧 使用设备: {device}")

print("📦 加载CLIP模型...")
clip_model, _ = clip.load("ViT-B/32", device=device, jit=False)
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad = False

# ---------- 6. 预先生成所有类别的CLIP文本嵌入 ----------
class_names = ['1', '2', '3', '4', '7']
text_descriptions = [f"an image of class {c}" for c in class_names]  # CLIP提示模板
text_tokens = clip.tokenize(text_descriptions).to(device)

with torch.no_grad():
    text_features = clip_model.encode_text(text_tokens)  # [5, 512]
    text_features = F.normalize(text_features, dim=-1)

print(f"✅ CLIP文本特征已生成，形状: {text_features.shape}")

# ---------- 7. 预训练数据集：ECoG + 对应文本特征 ----------
class ECogCLIPDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.FloatTensor(data)
        # 将原始标签映射到0-4
        self.labels = torch.LongTensor([mapping[l] for l in labels])
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

pretrain_dataset = ECogCLIPDataset(pretrain_data, pretrain_labels)
pretrain_loader = DataLoader(pretrain_dataset, batch_size=64, shuffle=True, drop_last=True)

# ---------- 8. ECoG编码器（适配CLIP特征维度512）----------
class ECoGEncoderCLIP(nn.Module):
    """将ECoG信号编码到CLIP共享语义空间（512维）"""
    def __init__(self, input_channels=72, time_points=450, output_dim=512):
        super().__init__()
        # 空间卷积
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool1 = nn.MaxPool1d(2)  # 450 -> 225
        
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.pool2 = nn.MaxPool1d(2)  # 225 -> 112
        
        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(256)
        self.pool3 = nn.MaxPool1d(2)  # 112 -> 56
        
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, output_dim)
        )
        
    def forward(self, x):
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool3(torch.relu(self.bn3(self.conv3(x))))
        x = self.gap(x).squeeze(-1)
        x = self.fc(x)
        return F.normalize(x, dim=-1)  # L2归一化，与CLIP特征对齐

# ---------- 9. 对比损失（ECoG特征 ↔ CLIP文本特征）----------
def clip_contrastive_loss(ecog_features, labels, text_features):
    """
    ecog_features: [B, 512] 已归一化
    labels: [B] 类别索引 (0-4)
    text_features: [5, 512] 已归一化
    """
    # 计算相似度矩阵 [B, 5]
    logits = ecog_features @ text_features.T / 0.07  # 温度系数
    
    # 每个样本的正样本是对应类别的文本特征
    loss = F.cross_entropy(logits, labels)
    
    # 可选：计算准确率作为监控
    with torch.no_grad():
        pred = logits.argmax(dim=-1)
        acc = (pred == labels).float().mean()
    
    return loss, acc

# ---------- 10. 初始化编码器 ----------
encoder = ECoGEncoderCLIP().to(device)
optimizer = optim.AdamW(encoder.parameters(), lr=1e-3, weight_decay=1e-4)

# ========== 11. ECoG-CLIP跨模态对齐预训练 ==========
ssl_epochs = 30
ssl_losses = []
ssl_accs = []

print("\n🚀 启动ECoG-CLIP跨模态对齐预训练...")
for epoch in range(ssl_epochs):
    encoder.train()
    total_loss = 0
    total_acc = 0
    
    pbar = tqdm(pretrain_loader, desc=f'CLIP Epoch {epoch+1}/{ssl_epochs}', leave=False)
    for batch_x, batch_y in pbar:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # 前向：ECoG -> 特征
        ecog_features = encoder(batch_x)
        
        # 对比损失：ECoG特征 vs 固定文本特征
        loss, acc = clip_contrastive_loss(ecog_features, batch_y, text_features)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        total_acc += acc.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{acc.item():.2f}'})
    
    avg_loss = total_loss / len(pretrain_loader)
    avg_acc = total_acc / len(pretrain_loader)
    ssl_losses.append(avg_loss)
    ssl_accs.append(avg_acc)
    print(f'Epoch {epoch+1:2d}: 平均损失 {avg_loss:.4f}, 对齐准确率 {avg_acc:.2f}')

# 绘制预训练曲线
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(ssl_losses)
plt.title('CLIP Contrastive Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(ssl_accs)
plt.title('ECoG→Text Alignment Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

# 保存预训练编码器
torch.save(encoder.state_dict(), 'ecog_clip_encoder.pth')
print("\n✅ ECoG-CLIP编码器已保存")

# ========== 12. 下游分类微调（740样本，包含所有被试）==========
class CLIPClassifier(nn.Module):
    def __init__(self, encoder, num_classes=5):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(512, num_classes)  # 直接在CLIP特征空间分类
        
    def forward(self, x):
        features = self.encoder(x)  # [B, 512]
        return self.classifier(features)

# 加载预训练编码器
finetune_encoder = ECoGEncoderCLIP().to(device)
finetune_encoder.load_state_dict(torch.load('ecog_clip_encoder.pth', map_location=device))
model = CLIPClassifier(finetune_encoder).to(device)

# 类别权重（可选，用于处理不平衡）
class_counts = np.bincount(finetune_y.numpy())
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)
criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(class_weights).to(device))

# ---------- 两阶段微调 ----------
# 第一阶段：冻结编码器，训练分类头
print("\n🎯 第一阶段：训练分类头（冻结编码器）")
for param in model.encoder.parameters():
    param.requires_grad = False

optimizer_head = optim.AdamW(model.classifier.parameters(), lr=1e-3)
finetune_loader = DataLoader(TensorDataset(finetune_X, finetune_y), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(val_X, val_y), batch_size=64, shuffle=False)

stage1_epochs = 10
best_val_acc = 0

for epoch in range(stage1_epochs):
    model.train()
    correct, total = 0, 0
    for batch_x, batch_y in finetune_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        optimizer_head.zero_grad()
        loss.backward()
        optimizer_head.step()
        _, pred = torch.max(outputs, 1)
        correct += (pred == batch_y).sum().item()
        total += batch_y.size(0)
    train_acc = 100 * correct / total
    
    # 验证
    model.eval()
    correct_val, total_val = 0, 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            _, pred = torch.max(outputs, 1)
            correct_val += (pred == batch_y).sum().item()
            total_val += batch_y.size(0)
    val_acc = 100 * correct_val / total_val
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_clip_classifier.pth')
    
    print(f"Stage1 Epoch {epoch+1:2d}: Train Acc {train_acc:.2f}%, Val Acc {val_acc:.2f}%")

# 第二阶段：全参数微调（低学习率）
print("\n🎯 第二阶段：全参数微调")
for param in model.encoder.parameters():
    param.requires_grad = True

optimizer_full = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer_full, mode='min', patience=3, factor=0.5)

stage2_epochs = 20

for epoch in range(stage2_epochs):
    model.train()
    correct, total = 0, 0
    for batch_x, batch_y in finetune_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        optimizer_full.zero_grad()
        loss.backward()
        optimizer_full.step()
        _, pred = torch.max(outputs, 1)
        correct += (pred == batch_y).sum().item()
        total += batch_y.size(0)
    train_acc = 100 * correct / total
    
    model.eval()
    correct_val, total_val = 0, 0
    val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            _, pred = torch.max(outputs, 1)
            correct_val += (pred == batch_y).sum().item()
            total_val += batch_y.size(0)
    val_acc = 100 * correct_val / total_val
    avg_val_loss = val_loss / len(val_loader)
    
    scheduler.step(avg_val_loss)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_clip_classifier.pth')
    
    print(f"Stage2 Epoch {epoch+1:2d}: Train Acc {train_acc:.2f}%, Val Acc {val_acc:.2f}%")

print(f"\n🏆 最佳验证准确率: {best_val_acc:.2f}%")

# ========== 13. 测试集评估 ==========
model.load_state_dict(torch.load('best_clip_classifier.pth', map_location=device))
model.eval()

test_loader = DataLoader(TensorDataset(test_X, test_y), batch_size=64, shuffle=False)
all_preds, all_true = [], []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x)
        _, pred = torch.max(outputs, 1)
        all_preds.extend(pred.cpu().numpy())
        all_true.extend(batch_y.cpu().numpy())

# 还原原始标签
reverse_mapping = {0:1, 1:2, 2:3, 3:4, 4:7}
orig_true = [reverse_mapping[l] for l in all_true]
orig_pred = [reverse_mapping[l] for l in all_preds]

# 混淆矩阵
cm = confusion_matrix(orig_true, orig_pred, labels=[1,2,3,4,7])
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['1','2','3','4','7'],
            yticklabels=['1','2','3','4','7'])
plt.title('Confusion Matrix (ECoG-CLIP)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

print("\n===== 测试集分类报告 =====")
print(classification_report(orig_true, orig_pred, target_names=['1','2','3','4','7']))
test_acc = np.mean(np.array(orig_true) == np.array(orig_pred))
print(f"✅ ECoG-CLIP最终测试集准确率: {test_acc*100:.2f}%")

torch.save(model.state_dict(), 'ecog_clip_final.pth')
print("\n✅ 最终模型已保存为 ecog_clip_final.pth")